# 03: Ratios and profit/loss (Phase 3)

**Goal:** compute the same ratio set for every company and label year, build a profit and loss summary for the four case companies, and check the case-company figures against their 10-K filings.

**Input:** `data/processed/statements_clean.csv` (Phase 2, USD millions).

**Outputs:**
- `data/processed/ratios.csv`: statement items plus all ratios and flags, one row per company and label year (2014 rows exist only as the growth base; `analysis_year` marks 2015 to 2025)
- `data/processed/pnl_case_companies.csv`: profit and loss summary for Nvidia, AMD, Intel and Wolfspeed
- `data/processed/tenk_verification.csv`: computed values vs the 10-K statements, with filing links

Formulas and denominator rules are in `src/ratios.py` and `report/drafts/ratio_definitions.md`. No winsorising here: outlier cutoffs must be fitted on training years only (Phase 5).

In [1]:
import os
import sys
import pandas as pd

sys.path.insert(0, "..")
from src import config
from src import ratios as R
from src import verify as V

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

statements = pd.read_csv(config.PROCESSED_DIR / "statements_clean.csv", parse_dates=["fiscal_year_end", "filed"])
print(f"Statements: {len(statements):,} company-years, {statements['cik'].nunique()} companies, "
      f"label years {statements['label_year'].min()}-{statements['label_year'].max()}")

Statements: 1,028 company-years, 161 companies, label years 2014-2025


## Step 1: Compute all ratios

| Category | Ratio | Formula |
| --- | --- | --- |
| Liquidity | Current ratio | current assets / current liabilities |
| Liquidity | Quick ratio | (current assets − inventory) / current liabilities |
| Liquidity | Cash ratio | cash / current liabilities |
| Leverage | Debt-to-equity | total liabilities / total equity |
| Leverage | Debt ratio | total liabilities / total assets |
| Leverage | Interest coverage | operating income / interest expense |
| Profitability | Gross, operating, net margin | gross profit, operating income, net income / revenue |
| Profitability | ROA | net income / total assets |
| Profitability | ROE | net income / total equity |
| Efficiency | Asset turnover | revenue / total assets |
| Growth | Revenue growth | revenue this year / last year − 1 |
| Growth | Net income growth | (this year − last year) / \|last year\| |
| Size | log total assets | ln(total assets in USD millions), for modelling |

Zero or negative denominators give a blank, never a number (rules in the next cell's output).

In [2]:
ratios = R.compute_ratios(statements)
ratios["analysis_year"] = ratios["label_year"].between(config.LABEL_YEAR_START, config.LABEL_YEAR_END)

# Why each ratio is blank: count rows by reason.
reasons = pd.DataFrame({
    "blank_rows": ratios[R.RATIO_COLUMNS].isna().sum(),
}).T
print(reasons.to_string())
print()
print("current liabilities missing:", ratios["current_liabilities"].isna().sum())
print("inventory missing (quick ratio blank):", ratios["inventory"].isna().sum())
print("cash missing:", ratios["cash"].isna().sum())
print("equity <= 0 (D/E and ROE blank, flag negative_equity):", ratios["negative_equity"].sum())
print("interest = 0, no debt (coverage blank, flag no_interest_expense):", ratios["no_interest_expense"].sum())
print("interest not available (coverage blank):", ratios["interest_expense"].isna().sum())
print("coverage computed from cash interest paid (flag coverage_uses_proxy):", ratios["coverage_uses_proxy"].sum())
print("revenue <= 0 (margins blank):", (ratios["revenue"] <= 0).sum())
print("first year of a company or a gap year (growth blank):",
      (~ratios.groupby("cik")["label_year"].diff().eq(1)).sum())

            current_ratio  quick_ratio  cash_ratio  debt_to_equity  debt_ratio  interest_coverage  gross_margin  operating_margin  net_margin  roa  roe  asset_turnover  revenue_growth  net_income_growth  log_total_assets
blank_rows              1           23           9              70           0                286            35                23           9    0   70               0             172                165                 0

current liabilities missing: 1
inventory missing (quick ratio blank): 22
cash missing: 8
equity <= 0 (D/E and ROE blank, flag negative_equity): 70
interest = 0, no debt (coverage blank, flag no_interest_expense): 222
interest not available (coverage blank): 50
coverage computed from cash interest paid (flag coverage_uses_proxy): 182
revenue <= 0 (margins blank): 9
first year of a company or a gap year (growth blank): 165


## Step 2: Distribution of each ratio (analysis years 2015 to 2025, all companies)

Medians are robust to the extreme values that tiny shell companies produce; the 1st and 99th percentiles show how heavy the tails are before winsorising in Phase 5.

In [3]:
a = ratios[ratios["analysis_year"]]
dist = a[R.RATIO_COLUMNS].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
dist[["count", "1%", "25%", "50%", "75%", "99%", "min", "max"]]

,count,1%,25%,50%,75%,99%,min,max
current_ratio,924.00,0.09,1.99,3.15,5.35,28.09,0.00,43.19
quick_ratio,904.00,0.03,1.42,2.40,4.45,24.57,0.00,38.31
cash_ratio,918.00,0.01,0.57,1.07,2.05,13.22,0.00,42.68
debt_to_equity,861.00,0.03,0.25,0.56,1.13,6.56,0.01,211.59
debt_ratio,925.00,0.03,0.21,0.38,0.56,4.49,0.01,"2,303.28"
interest_coverage,675.00,"-1,781.59",-8.57,2.26,15.30,447.46,"-77,639.04","5,753.86"
gross_margin,892.00,-0.60,0.28,0.44,0.57,0.97,-11.18,1.68
operating_margin,908.00,-127.08,-0.24,0.02,0.15,0.60,-905.67,261.72
net_margin,917.00,-125.19,-0.24,0.01,0.14,0.64,-951.44,259.28
roa,925.00,-5.62,-0.16,0.01,0.08,0.33,"-1,581.24","17,617.96"


In [4]:
# Sector medians by year: how the typical semiconductor company changed.
med = a.groupby("label_year")[["current_ratio", "debt_ratio", "gross_margin", "operating_margin", "net_margin",
                               "roa", "revenue_growth"]].median()
med["companies"] = a.groupby("label_year")["cik"].nunique()
med["share_with_net_loss"] = a.groupby("label_year")["net_income"].apply(lambda s: (s < 0).mean())
med

,current_ratio,debt_ratio,gross_margin,operating_margin,net_margin,roa,revenue_growth,companies,share_with_net_loss
label_year,,,,,,,,,
2015,3.40,0.30,0.43,0.01,0.01,0.00,-0.01,93,0.49
2016,3.55,0.38,0.47,0.04,0.02,0.01,0.05,83,0.47
2017,3.40,0.41,0.47,0.07,0.03,0.01,0.12,86,0.48
2018,3.32,0.36,0.43,0.02,0.01,0.00,0.07,85,0.48
2019,2.81,0.43,0.44,-0.00,0.00,0.00,-0.03,83,0.49
2020,3.18,0.42,0.45,0.05,0.04,0.03,0.06,81,0.41
2021,3.13,0.39,0.45,0.09,0.06,0.04,0.26,89,0.43
2022,2.98,0.41,0.43,0.05,0.06,0.02,0.14,83,0.42
2023,3.21,0.36,0.40,0.00,-0.02,-0.01,-0.06,81,0.54


## Step 3: The four case companies

Key ratios by label year. Blanks follow the denominator rules (e.g. AMD 2015 and Wolfspeed 2025 have negative equity; Wolfspeed 2014 reports no interest expense).

In [5]:
case = ratios[ratios["cik"].isin(config.CASE_COMPANIES)].copy()
case["company"] = case["cik"].map(config.CASE_COMPANIES)
show = ["current_ratio", "quick_ratio", "cash_ratio", "debt_to_equity", "debt_ratio", "interest_coverage",
        "gross_margin", "operating_margin", "net_margin", "roa", "roe", "asset_turnover", "revenue_growth", "net_income_growth"]
for company in config.CASE_COMPANIES.values():
    c = case[case["company"] == company].set_index("label_year")
    print(f"=== {company}  (flags: negative_equity in {c.index[c['negative_equity']].tolist()}, "
          f"coverage from cash-paid proxy in {c.index[c['coverage_uses_proxy']].tolist()}, "
          f"no interest expense in {c.index[c['no_interest_expense']].tolist()})")
    display(c[show])

=== Nvidia  (flags: negative_equity in [], coverage from cash-paid proxy in [], no interest expense in [])


,current_ratio,quick_ratio,cash_ratio,debt_to_equity,debt_ratio,interest_coverage,gross_margin,operating_margin,net_margin,roa,roe,asset_turnover,revenue_growth,net_income_growth
label_year,,,,,,,,,,,,,,
2014,6.38,5.84,0.55,0.63,0.39,16.45,0.56,0.16,0.13,0.09,0.14,0.65,NaN,NaN
2015,2.57,2.40,0.25,0.63,0.38,15.89,0.56,0.15,0.12,0.08,0.14,0.68,0.07,-0.03
2016,4.77,4.33,0.99,0.70,0.41,33.34,0.59,0.28,0.24,0.17,0.29,0.70,0.38,1.71
2017,8.03,7.34,3.47,0.50,0.34,52.62,0.60,0.33,0.31,0.27,0.41,0.86,0.41,0.83
2018,7.94,6.76,0.59,0.42,0.30,65.59,0.61,0.32,0.35,0.31,0.44,0.88,0.21,0.36
2019,7.67,7.12,6.11,0.42,0.30,54.73,0.62,0.26,0.26,0.16,0.23,0.63,-0.07,-0.32
2020,4.09,3.63,0.22,0.70,0.41,24.63,0.62,0.27,0.26,0.15,0.26,0.58,0.53,0.55
2021,6.65,6.05,0.46,0.66,0.40,42.55,0.65,0.37,0.36,0.22,0.37,0.61,0.61,1.25
2022,3.52,2.73,0.52,0.86,0.46,16.12,0.57,0.16,0.16,0.11,0.20,0.65,0.00,-0.55


=== AMD  (flags: negative_equity in [2015], coverage from cash-paid proxy in [], no interest expense in [])


,current_ratio,quick_ratio,cash_ratio,debt_to_equity,debt_ratio,interest_coverage,gross_margin,operating_margin,net_margin,roa,roe,asset_turnover,revenue_growth,net_income_growth
label_year,,,,,,,,,,,,,,
2014,1.90,1.42,0.56,19.14,0.95,-0.88,0.33,-0.03,-0.07,-0.11,-2.16,1.46,NaN,NaN
2015,1.65,1.17,0.56,NaN,1.13,-3.01,0.27,-0.12,-0.17,-0.21,NaN,1.28,-0.28,-0.64
2016,1.88,1.32,0.94,6.98,0.87,-2.38,0.23,-0.09,-0.12,-0.15,-1.19,1.29,0.07,0.25
2017,1.76,1.27,0.80,4.79,0.83,1.62,0.34,0.04,0.01,0.01,0.07,1.51,0.25,1.09
2018,1.78,1.36,0.54,2.60,0.72,3.73,0.38,0.07,0.05,0.07,0.27,1.42,0.22,6.84
2019,1.95,1.53,0.62,1.13,0.53,6.71,0.43,0.09,0.05,0.06,0.12,1.12,0.04,0.01
2020,2.54,1.96,0.66,0.54,0.35,29.13,0.45,0.14,0.26,0.28,0.43,1.09,0.45,6.30
2021,2.02,1.56,0.60,0.66,0.40,107.29,0.48,0.22,0.19,0.25,0.42,1.32,0.68,0.27
2022,2.36,1.77,0.76,0.23,0.19,14.36,0.45,0.05,0.06,0.02,0.02,0.35,0.44,-0.58


=== Intel  (flags: negative_equity in [], coverage from cash-paid proxy in [], no interest expense in [])


,current_ratio,quick_ratio,cash_ratio,debt_to_equity,debt_ratio,interest_coverage,gross_margin,operating_margin,net_margin,roa,roe,asset_turnover,revenue_growth,net_income_growth
label_year,,,,,,,,,,,,,,
2014,1.73,1.46,0.16,0.63,0.38,79.93,0.64,0.27,0.21,0.13,0.21,0.61,NaN,NaN
2015,2.58,2.25,0.98,0.67,0.40,41.55,0.63,0.25,0.21,0.11,0.19,0.54,-0.01,-0.02
2016,1.75,1.48,0.27,0.70,0.41,17.56,0.61,0.22,0.17,0.09,0.16,0.52,0.07,-0.10
2017,1.69,1.29,0.20,0.77,0.43,27.76,0.62,0.29,0.15,0.08,0.14,0.51,0.06,-0.07
2018,1.73,1.30,0.18,0.71,0.41,49.82,0.62,0.33,0.30,0.16,0.28,0.55,0.13,1.19
2019,1.40,1.01,0.19,0.76,0.43,45.06,0.59,0.31,0.29,0.15,0.27,0.53,0.02,-0.00
2020,1.91,1.57,0.24,0.89,0.47,37.64,0.56,0.30,0.27,0.14,0.26,0.51,0.08,-0.01
2021,2.10,1.71,0.18,0.77,0.43,32.59,0.55,0.25,0.25,0.12,0.21,0.47,0.01,-0.05
2022,1.57,1.16,0.35,0.78,0.43,4.71,0.43,0.04,0.13,0.04,0.08,0.35,-0.20,-0.60


=== Wolfspeed  (flags: negative_equity in [2025], coverage from cash-paid proxy in [2015, 2016, 2017, 2018, 2019], no interest expense in [2014])


,current_ratio,quick_ratio,cash_ratio,debt_to_equity,debt_ratio,interest_coverage,gross_margin,operating_margin,net_margin,roa,roe,asset_turnover,revenue_growth,net_income_growth
label_year,,,,,,,,,,,,,,
2014,5.78,4.86,0.94,0.12,0.11,NaN,0.38,0.08,0.08,0.04,0.04,0.49,NaN,NaN
2015,5.13,4.03,0.55,0.20,0.17,-72.37,0.29,-0.04,-0.04,-0.02,-0.03,0.55,-0.01,-1.52
2016,5.19,3.83,0.75,0.17,0.14,-3.37,0.30,-0.01,-0.01,-0.01,-0.01,0.58,-0.01,0.66
2017,5.19,3.85,0.63,0.19,0.16,-5.20,0.30,-0.01,-0.07,-0.04,-0.04,0.56,-0.09,-3.56
2018,3.58,2.39,0.48,0.27,0.21,-54.01,0.27,-0.22,-0.19,-0.11,-0.14,0.57,0.01,-1.85
2019,5.27,4.57,1.87,0.38,0.28,-3.98,0.36,-0.01,-0.35,-0.13,-0.18,0.38,-0.28,-0.34
2020,5.47,4.85,1.54,0.55,0.35,-6.00,0.27,-0.23,-0.21,-0.06,-0.09,0.28,-0.16,0.49
2021,3.29,2.92,0.84,0.63,0.39,-6.91,0.31,-0.60,-1.00,-0.15,-0.25,0.15,-0.42,-1.73
2022,4.54,3.95,1.16,0.61,0.38,-9.87,0.33,-0.33,-0.27,-0.05,-0.08,0.19,0.42,0.62


## Step 4: Profit and loss summary (case companies)

Reported gross profit and operating income are used as filed. The lines between them are derived so each statement adds up:
- operating expenses = gross profit − operating income
- other non-operating items = pre-tax income − (operating income − interest expense)
- discontinued operations and other = net income − (pre-tax income − income tax)

USD millions; expenses shown as positive amounts.

In [6]:
pnl = R.profit_and_loss(statements[statements["cik"].isin(config.CASE_COMPANIES)])
pnl["company"] = pnl["cik"].map(config.CASE_COMPANIES)
pnl = pnl[["company"] + [c for c in pnl.columns if c != "company"]]

# Consistency check: where cost of revenue is reported, revenue - cost should equal reported gross profit.
diff = (pnl["revenue"] - pnl["cost_of_revenue"] - pnl["gross_profit"]).abs()
print("Years where revenue - cost of revenue differs from reported gross profit by > 1:",
      pnl.loc[diff > 1, ["company", "label_year"]].values.tolist())

pnl.to_csv(config.PROCESSED_DIR / "pnl_case_companies.csv", index=False)
print("Saved", config.PROCESSED_DIR / "pnl_case_companies.csv")

Years where revenue - cost of revenue differs from reported gross profit by > 1: []
Saved G:\DJSCE\Fourth Year\FADS\Mini Proj\data\processed\pnl_case_companies.csv


In [7]:
lines = ["revenue", "cost_of_revenue", "gross_profit", "operating_expenses", "operating_income", "interest_expense",
         "other_nonoperating", "pretax_income", "income_tax", "discontinued_and_other", "net_income",
         "profit_or_loss", "revenue_change_pct", "net_income_change"]
for company in config.CASE_COMPANIES.values():
    print(f"=== {company} (USD millions)")
    display(pnl[pnl["company"] == company].set_index("label_year")[lines].T)

=== Nvidia (USD millions)


label_year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
revenue,"4,681.51","5,010.00","6,910.00","9,714.00","11,716.00","10,918.00","16,675.00","26,914.00","26,974.00","60,922.00","130,497.00","215,938.00"
cost_of_revenue,"2,082.03","2,199.00","2,847.00","3,892.00","4,545.00","4,150.00","6,279.00","9,439.00","11,618.00","16,621.00","32,639.00","62,475.00"
gross_profit,"2,599.48","2,811.00","4,063.00","5,822.00","7,171.00","6,768.00","10,396.00","17,475.00","15,356.00","44,301.00","97,858.00","153,463.00"
operating_expenses,"1,840.49","2,064.00","2,129.00","2,612.00","3,367.00","3,922.00","5,864.00","7,434.00","11,132.00","11,329.00","16,405.00","23,076.00"
operating_income,758.99,747.00,"1,934.00","3,210.00","3,804.00","2,846.00","4,532.00","10,041.00","4,224.00","32,972.00","81,453.00","130,387.00"
interest_expense,46.13,47.00,58.00,61.00,58.00,52.00,184.00,236.00,262.00,257.00,247.00,259.00
other_nonoperating,41.98,43.00,29.00,47.00,150.00,176.00,61.00,136.00,219.00,"1,103.00","2,820.00","11,322.00"
pretax_income,754.84,743.00,"1,905.00","3,196.00","3,896.00","2,970.00","4,409.00","9,941.00","4,181.00","33,818.00","84,026.00","141,450.00"
income_tax,124.25,129.00,239.00,149.00,-245.00,174.00,77.00,189.00,-187.00,"4,058.00","11,146.00","21,383.00"
discontinued_and_other,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


=== AMD (USD millions)


label_year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
revenue,"5,506.00","3,991.00","4,272.00","5,329.00","6,475.00","6,731.00","9,763.00","16,434.00","23,601.00","22,680.00","25,785.00","34,639.00"
cost_of_revenue,"3,667.00","2,911.00","3,274.00","3,506.00","4,028.00","3,863.00","5,416.00","8,505.00","12,998.00","12,220.00","13,060.00","17,487.00"
gross_profit,"1,839.00","1,080.00",998.00,"1,823.00","2,447.00","2,868.00","4,347.00","7,929.00","10,603.00","10,460.00","12,725.00","17,152.00"
operating_expenses,"1,994.00","1,561.00","1,370.00","1,619.00","1,996.00","2,237.00","2,978.00","4,281.00","9,339.00","10,059.00","10,825.00","13,458.00"
operating_income,-155.00,-481.00,-372.00,204.00,451.00,631.00,"1,369.00","3,648.00","1,264.00",401.00,"1,900.00","3,694.00"
interest_expense,177.00,160.00,156.00,126.00,121.00,94.00,47.00,34.00,88.00,106.00,92.00,131.00
other_nonoperating,-66.00,-5.00,80.00,-9.00,0.00,-165.00,-47.00,55.00,8.00,197.00,181.00,577.00
pretax_income,-398.00,-646.00,-448.00,69.00,330.00,372.00,"1,275.00","3,669.00","1,184.00",492.00,"1,989.00","4,140.00"
income_tax,5.00,14.00,39.00,19.00,-9.00,31.00,"-1,210.00",513.00,-122.00,-346.00,381.00,-103.00
discontinued_and_other,0.00,0.00,-10.00,-7.00,-2.00,0.00,5.00,6.00,14.00,16.00,33.00,92.00


=== Intel (USD millions)


label_year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
revenue,"55,870.00","55,355.00","59,387.00","62,761.00","70,848.00","71,965.00","77,867.00","79,024.00","63,054.00","54,228.00","53,101.00","52,853.00"
cost_of_revenue,"20,261.00","20,676.00","23,196.00","23,692.00","27,111.00","29,825.00","34,255.00","35,209.00","36,188.00","32,517.00","35,756.00","34,478.00"
gross_profit,"35,609.00","34,679.00","36,191.00","39,069.00","43,737.00","42,140.00","43,612.00","43,815.00","26,866.00","21,711.00","17,345.00","18,375.00"
operating_expenses,"20,262.00","20,677.00","23,317.00","21,133.00","20,421.00","20,105.00","19,934.00","24,359.00","24,532.00","21,618.00","29,023.00","20,589.00"
operating_income,"15,347.00","14,002.00","12,874.00","17,936.00","23,316.00","22,035.00","23,678.00","19,456.00","2,334.00",93.00,"-11,678.00","-2,214.00"
interest_expense,192.00,337.00,733.00,646.00,468.00,489.00,629.00,597.00,496.00,878.00,"1,034.00","1,091.00"
other_nonoperating,646.00,547.00,795.00,"3,062.00",469.00,"2,512.00","2,029.00","2,844.00","5,930.00","1,547.00","1,502.00","4,862.00"
pretax_income,"15,801.00","14,212.00","12,936.00","20,352.00","23,317.00","24,058.00","25,078.00","21,703.00","7,768.00",762.00,"-11,210.00","1,557.00"
income_tax,"4,097.00","2,792.00","2,620.00","10,751.00","2,264.00","3,010.00","4,179.00","1,835.00",-249.00,-913.00,"8,023.00","1,531.00"
discontinued_and_other,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-3.00,14.00,477.00,-293.00


=== Wolfspeed (USD millions)


label_year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
revenue,"1,647.64","1,632.51","1,616.63","1,473.00","1,493.68","1,080.00",903.90,525.60,746.20,921.90,807.20,757.60
cost_of_revenue,"1,028.85","1,157.55","1,129.55","1,038.43","1,086.04",689.00,655.60,361.00,496.90,642.40,729.80,879.20
gross_profit,618.79,474.96,487.07,434.57,407.64,391.00,248.30,164.60,249.30,279.50,77.40,-121.60
operating_expenses,484.52,547.47,497.55,453.24,736.73,406.90,457.70,478.50,497.10,660.10,522.70,"1,207.60"
operating_income,134.28,-72.51,-10.47,-18.67,-329.09,-15.90,-209.40,-313.90,-247.80,-380.60,-445.30,"-1,329.20"
interest_expense,0.00,1.00,3.11,3.59,6.09,4.00,34.90,45.40,25.10,42.60,246.30,315.20
other_nonoperating,13.29,-9.39,-9.93,17.60,17.74,-25.30,53.90,19.10,-13.20,94.70,119.10,25.50
pretax_income,147.57,-82.90,-23.51,-4.66,-317.44,-45.20,-190.40,-340.20,-286.10,-328.50,-572.50,"-1,618.90"
income_tax,23.38,-18.85,-1.97,93.45,-37.52,12.70,0.20,1.10,9.00,1.40,1.10,-9.70
discontinued_and_other,0.00,0.00,0.00,0.00,-0.05,-317.20,-1.10,-182.60,94.20,0.00,-290.60,0.00


In [8]:
# Profit and loss years at a glance.
pnl.pivot_table(index="label_year", columns="company", values="profit_or_loss", aggfunc="first")

company,AMD,Intel,Nvidia,Wolfspeed
label_year,,,,
2014,Loss,Profit,Profit,Profit
2015,Loss,Profit,Profit,Loss
2016,Loss,Profit,Profit,Loss
2017,Profit,Profit,Profit,Loss
2018,Profit,Profit,Profit,Loss
2019,Profit,Profit,Profit,Loss
2020,Profit,Profit,Profit,Loss
2021,Profit,Profit,Profit,Loss
2022,Profit,Profit,Profit,Loss


## Step 5: Check against the 10-K filings

For 2017 (before the ASC 606 revenue tag change) and 2024 for each company, plus Wolfspeed 2025 (filed during its Chapter 11 restructuring), the SEC's rendered balance sheet and income statement of the exact 10-K are downloaded from EDGAR (cached in `data/raw/edgar_statements/`). Each computed value is looked up among the statement lines (full-year column, scaled to USD millions); a match within rounding confirms the number and shows the 10-K line it came from.

Total liabilities is not checked directly because AMD, Intel and Wolfspeed do not report it on the balance sheet (it is derived); total liabilities and equity, the base of the derivation, is checked instead.

In [9]:
ua = os.environ.get(config.SEC_USER_AGENT_ENV, "")  # only needed if a page is not cached yet
checks = []
for cik, year in [(c, y) for c in config.CASE_COMPANIES for y in (2017, 2024)] + [(895419, 2025)]:
    row = statements[(statements["cik"] == cik) & (statements["label_year"] == year)].iloc[0]
    paths = V.download_statements(cik, row["adsh"], config.RAW_DIR / "edgar_statements", ua)
    res = V.check_filing(row, paths)
    res.insert(0, "label_year", year)
    res.insert(0, "company", config.CASE_COMPANIES[cik])
    res["filing"] = V.filing_url(cik, row["adsh"], "")
    checks.append(res)
checks = pd.concat(checks, ignore_index=True)
checks.to_csv(config.PROCESSED_DIR / "tenk_verification.csv", index=False)

print("Matches per filing:")
print(checks.groupby(["company", "label_year"])["status"].apply(lambda s: f"{(s == 'match').sum()}/{len(s)}").to_string())
print()
print("Mismatches:", (checks["status"] != "match").sum())
checks[checks["status"] != "match"]

Matches per filing:
company    label_year
AMD        2017          11/11
           2024          11/11
Intel      2017          11/11
           2024          11/11
Nvidia     2017          11/11
           2024          11/11
Wolfspeed  2017          11/11
           2024          11/11
           2025          11/11

Mismatches: 0


,company,label_year,item,computed_musd,statement,status,tenk_line,tenk_value_musd,filing


In [10]:
# Full detail: computed value, the 10-K line it matched and that line's value.
checks[["company", "label_year", "item", "computed_musd", "tenk_line", "tenk_value_musd", "status"]]

,company,label_year,item,computed_musd,tenk_line,tenk_value_musd,status
0,Nvidia,2017,revenue,"9,714.00",Revenue,"9,714.00",match
1,Nvidia,2017,gross_profit,"5,822.00",Gross profit,"5,822.00",match
2,Nvidia,2017,operating_income,"3,210.00",Income from operations,"3,210.00",match
3,Nvidia,2017,net_income,"3,047.00",Net income,"3,047.00",match
4,Nvidia,2017,total_assets,"11,241.00",Total assets,"11,241.00",match
5,Nvidia,2017,current_assets,"9,255.00",Total current assets,"9,255.00",match
6,Nvidia,2017,current_liabilities,"1,153.00",Total current liabilities,"1,153.00",match
7,Nvidia,2017,liabilities_and_equity,"11,241.00",Total assets,"11,241.00",match
8,Nvidia,2017,equity,"7,471.00",Total shareholders' equity,"7,471.00",match
9,Nvidia,2017,cash,"4,002.00",Cash and cash equivalents,"4,002.00",match


In [11]:
# Filing links for a manual spot check (open the link, then the 10-K document).
checks.groupby(["company", "label_year"])["filing"].first().to_frame()

filing
company   label_year                                                   
AMD       2017        https://www.sec.gov/Archives/edgar/data/2488/0...
          2024        https://www.sec.gov/Archives/edgar/data/2488/0...
Intel     2017        https://www.sec.gov/Archives/edgar/data/50863/...
          2024        https://www.sec.gov/Archives/edgar/data/50863/...
Nvidia    2017        https://www.sec.gov/Archives/edgar/data/104581...
          2024        https://www.sec.gov/Archives/edgar/data/104581...
Wolfspeed 2017        https://www.sec.gov/Archives/edgar/data/895419...
          2024        https://www.sec.gov/Archives/edgar/data/895419...
          2025        https://www.sec.gov/Archives/edgar/data/895419...

## Step 6: Save the ratios table

In [12]:
ratios.to_csv(config.PROCESSED_DIR / "ratios.csv", index=False)
print(f"Saved ratios.csv: {len(ratios):,} rows x {ratios.shape[1]} columns "
      f"({ratios['analysis_year'].sum():,} rows in analysis years {config.LABEL_YEAR_START}-{config.LABEL_YEAR_END})")

Saved ratios.csv: 1,028 rows x 62 columns (925 rows in analysis years 2015-2025)
